Connect the drive first and foremost (no matter what!)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Install and import

In [ ]:
!pip install albumentations -q

import os, shutil, random, yaml
import cv2
import numpy as np
import albumentations as A
from pathlib import Path

In [ ]:
import zipfile

# Paths
ZIP_PATH    = '/content/drive/MyDrive/Dataset_hadundoda/fypWhiteTea.yolov8.zip'
EXTRACT_DIR = '/content/raw_dataset'
OUTPUT_DIR  = '/content/drive/MyDrive/augment_herman'
CLASS_NAMES = ['unopened_leaf']

# unzip
print('Unzipping...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)
print('Done.')

# what actually came out:
import os
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:3]:
            print(f'{indent}  {f}')

Unzipping...
Done.
raw_dataset/
  data.yaml
  README.roboflow.txt
  train/
    images/
    labels/


Augmentation pipeline

In [ ]:
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=20, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.2, p=0.8),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.5),
    A.GaussNoise(var_limit=(5, 25), p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.RandomScale(scale_limit=0.2, p=0.5),
    A.CLAHE(p=0.3),
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.3  # drop boxes if mostly cropped out
))

/tmp/ipykernel_2227/108325025.py:7: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5, 25), p=0.4),


Helper to read/write YOLO labels

In [ ]:
def read_yolo_labels(label_path):
    boxes, classes = [], []
    if not os.path.exists(label_path):
        return boxes, classes
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) == 5:
                classes.append(int(parts[0]))
                boxes.append([float(x) for x in parts[1:]])
    return boxes, classes

def write_yolo_labels(label_path, boxes, classes):
    with open(label_path, 'w') as f:
        for cls, box in zip(classes, boxes):
            f.write(f"{cls} {' '.join([f'{x:.6f}' for x in box])}\n")

Run Augmentation

In [ ]:
SOURCE_IMAGES = '/content/raw_dataset/train/images'
SOURCE_LABELS = '/content/raw_dataset/train/labels'

In [ ]:
AUGMENTATIONS_PER_IMAGE = 5  # gives ~360+ images from 72

# Create output folders
for split in ['train', 'valid', 'test']:
    os.makedirs(f'{OUTPUT_DIR}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUTPUT_DIR}/{split}/labels', exist_ok=True)

# Gather all images
all_images = sorted([f for f in os.listdir(SOURCE_IMAGES) if f.endswith('.jpg')])
random.seed(42)
random.shuffle(all_images)

# Split: 80/10/10
n = len(all_images)
splits = {
    'train': all_images[:int(n*0.8)],
    'valid': all_images[int(n*0.8):int(n*0.9)],
    'test':  all_images[int(n*0.9):]
}

for split, images in splits.items():
    print(f'\nProcessing {split}: {len(images)} original images')
    for img_file in images:
        img_path = os.path.join(SOURCE_IMAGES, img_file)
        label_path = os.path.join(SOURCE_LABELS, img_file.replace('.jpg', '.txt'))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        boxes, classes = read_yolo_labels(label_path)

        stem = Path(img_file).stem

        # Always copy original first
        out_img = f'{OUTPUT_DIR}/{split}/images/{stem}.jpg'
        out_lbl = f'{OUTPUT_DIR}/{split}/labels/{stem}.txt'
        cv2.imwrite(out_img, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
        write_yolo_labels(out_lbl, boxes, classes)

        # Only augment training split
        if split != 'train':
            continue

        for i in range(AUGMENTATIONS_PER_IMAGE):
            try:
                if boxes:
                    augmented = transform(image=image, bboxes=boxes, class_labels=classes)
                else:
                    aug_simple = A.Compose([
                        A.HorizontalFlip(p=0.5),
                        A.RandomBrightnessContrast(p=0.8),
                        A.Rotate(limit=20, p=0.7),
                    ])
                    augmented = aug_simple(image=image)
                    augmented['bboxes'] = []
                    augmented['class_labels'] = []

                aug_img = augmented['image']
                aug_boxes = augmented['bboxes']
                aug_classes = augmented['class_labels']

                out_img = f'{OUTPUT_DIR}/{split}/images/{stem}_aug{i}.jpg'
                out_lbl = f'{OUTPUT_DIR}/{split}/labels/{stem}_aug{i}.txt'
                cv2.imwrite(out_img, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
                write_yolo_labels(out_lbl, aug_boxes, aug_classes)

            except Exception as e:
                print(f'  Skipped {stem}_aug{i}: {e}')

print('\nDone!')


Processing train: 57 original images

Processing valid: 7 original images

Processing test: 8 original images

Done!


Generate data.yaml

In [ ]:
import yaml

yaml_content = {
    'path': OUTPUT_DIR,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES
}

with open(f'{OUTPUT_DIR}/data.yaml', 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f'data.yaml saved to {OUTPUT_DIR}/data.yaml')
print(yaml_content)

data.yaml saved to /content/drive/MyDrive/augment_herman/data.yaml
{'path': '/content/drive/MyDrive/augment_herman', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 1, 'names': ['unopened_leaf']}


Check the count

In [ ]:
for split in ['train', 'valid', 'test']:
    imgs = len(os.listdir(f'{OUTPUT_DIR}/{split}/images'))
    lbls = len(os.listdir(f'{OUTPUT_DIR}/{split}/labels'))
    print(f'{split}: {imgs} images, {lbls} labels')

print(f'\nFull augmented dataset saved to Drive: {OUTPUT_DIR}')

train: 342 images, 342 labels
valid: 7 images, 7 labels
test: 8 images, 8 labels

Full augmented dataset saved to Drive: /content/drive/MyDrive/augment_herman
